# The Claire Speech Engine: Engineering Validation
**Notebook:** COLAB-001
**Purpose:** Validate CSE installation, backend abstraction, and public API using remote evaluation backends (Fish Speech & StyleTTS2).

## 1. Environment & Dependencies
Install audio and tensor libraries needed for evaluation backends.

In [ ]:
!apt-get install -y -qq portaudio19-dev
!pip install torch torchaudio soundfile numpy

## 2. Install Fish Speech v1.5 (VRAM-optimized)
Clones the v1.5.0 tag, patches dependency pins, and installs via pip.

In [ ]:
import os
if not os.path.exists('fish-speech'):
    !git clone https://github.com/fishaudio/fish-speech.git
    !cd fish-speech && git checkout v1.5.0
else:
    print('fish-speech/ already cloned, skipping.')

# Download fish-speech 1.5 weights
!pip install -q -U huggingface_hub
!hf download fishaudio/fish-speech-1.5 --local-dir checkpoints/fish-speech-1.5

# CRITICAL FIX: Fish Speech pins old versions of tokenizers/transformers/torch
import re
for filepath in ['fish-speech/pyproject.toml', 'fish-speech/requirements.txt']:
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            content = f.read()
        for pkg in ['tokenizers', 'transformers', 'torch', 'numpy']:
            pattern = pkg + r'(?:\[[^\]]*\])?(?:\s*[><=!~]+\s*[\w.*]+(?:\s*,\s*[><=!~]+\s*[\w.*]+)*)'
            content = re.sub(pattern, pkg, content)
        with open(filepath, 'w') as f:
            f.write(content)
        print(f'Patched: {filepath}')

# Install Fish Speech from the cloned repo (--no-build-isolation lets it use existing torch)
!pip install -q --no-build-isolation -e fish-speech


## 3. Install CSE
Install The Claire Speech Engine directly from the main repository branch.

In [ ]:
!pip install git+https://github.com/Yukariii04/Claire-Speech-Engine.git

## 4. Fish Speech Validation
Initialize the engine, switch to the Fish Speech adapter, and generate a test audio file.

In [ ]:
from cse import SpeechEngine
from cse.voice import register_voice_package, VoicePackage, VoiceMetadata
from pathlib import Path
import os
from IPython.display import Audio, display
from google.colab import files

print("Initializing Engine...")
engine = SpeechEngine()

print("Registering mock 'default' voice...")
meta = VoiceMetadata(id="default", name="Default", version="1.0.0", author="CSE", language="en", backend="fishspeech", sample_rate=24000, channels=1, description="Mock voice", license="MIT")
try:
    register_voice_package(VoicePackage(metadata=meta, path=Path(".")))
except Exception:
    pass

print("Loading Fish Speech Backend...")
engine.load_backend("fishspeech")
engine.load_voice("default")

print("Generating Speech via Fish Speech...")
speech_fs = engine.speak("Testing the Fish Speech acoustic adapter.")

print(f"Success: {speech_fs.success}")
print(f"Audio saved to: {speech_fs.audio_path}")

if os.path.exists(speech_fs.audio_path):
    display(Audio(speech_fs.audio_path))
else:
    print("Audio file missing.")

## 5. StyleTTS2 Validation
Switch to the StyleTTS2 adapter seamlessly on the same engine instance and generate audio.

In [ ]:
print("Switching to StyleTTS2 Backend...")
engine.load_backend("styletts2")
engine.load_voice("default")

print("Generating Speech via StyleTTS2...")
speech_st = engine.speak("Testing the Style TTS 2 acoustic adapter.")

print(f"Success: {speech_st.success}")
print(f"Audio saved to: {speech_st.audio_path}")

if os.path.exists(speech_st.audio_path):
    display(Audio(speech_st.audio_path))
else:
    print("Audio file missing.")

## 6. Download Outputs
Download the validation WAV files to your local machine.

In [ ]:
if os.path.exists(speech_fs.audio_path):
    print("Downloading Fish Speech validation...")
    files.download(speech_fs.audio_path)

if os.path.exists(speech_st.audio_path):
    print("Downloading StyleTTS2 validation...")
    files.download(speech_st.audio_path)